In [3]:
# unzip file
from zipfile import ZipFile

fileName = 'archive'

with ZipFile("demo_data/"+fileName+".zip", 'r') as zObject:
    zObject.extractall(
      path="demo_data/")

In [12]:
#!conda install opencv -y

In [11]:
import os
import cv2
import numpy as np

from matplotlib import pyplot as plt
from patchify import patchify
from PIL import Image
#import segmentation_models as sm
from tensorflow.keras.metrics import MeanIoU

from sklearn.preprocessing import MinMaxScaler, StandardScaler

os.environ["SM_FRAMEWORK"] = "tf.keras"

from tensorflow import keras
import segmentation_models as sm

"""
Use patchify....
Tile 1: 797 x 644 --> 768 x 512 --> 6
Tile 2: 509 x 544 --> 512 x 256 --> 2
Tile 3: 682 x 658 --> 512 x 512  --> 4
Tile 4: 1099 x 846 --> 1024 x 768 --> 12
Tile 5: 1126 x 1058 --> 1024 x 1024 --> 16
Tile 6: 859 x 838 --> 768 x 768 --> 9
Tile 7: 1817 x 2061 --> 1792 x 2048 --> 56
Tile 8: 2149 x 1479 --> 1280 x 2048 --> 40
Total 9 images in each folder * (145 patches) = 1305
Total 1305 patches of size 256x256
"""


scaler = MinMaxScaler()

#root_directory = 'demo_data/Semantic segmentation dataset/'
#root_directory = 'satData/S2B_MSIL2A_20230612T073619_N0509_R092_T37NCC_20230612T101141\.SAFE/GRANULE/L2A_T37NCC_A032722_20230612T075442/IMG_DATA/R10m/'
#root_directory = 'bands'

patch_size = 256

In [12]:
root_directory = 'sat_Data/S2A_MSIL2A_20230627T073621_N0509_R092_T37NCC_20230627T110801.SAFE/GRANULE/L2A_T37NCC_A041845_20230627T075440/IMG_DATA/'
root_directory

'sat_Data/S2A_MSIL2A_20230627T073621_N0509_R092_T37NCC_20230627T110801.SAFE/GRANULE/L2A_T37NCC_A041845_20230627T075440/IMG_DATA/'

In [13]:
#Read images from repsective 'images' subdirectory
#As all images are of ddifferent size we have 2 options, either resize or crop
#But, some images are too large and some small. Resizing will change the size of real objects.
#Therefore, we will crop them to a nearest size divisible by 256 and then 
#divide all images into patches of 256x256x3. 
image_dataset = []  
for path, subdirs, files in os.walk(root_directory):
    print(path)  
    dirname = path.split(os.path.sep)[-1]
    if dirname == 'R10m':   #Find all 'images' directories
        images = os.listdir(path)  #List of all image names in this subdirectory
        for i, image_name in enumerate(images):  
            if image_name.endswith(".jp2"):   #Only read jpg images...
               
                image = cv2.imread(path+"/"+image_name, 1)  #Read each image as BGR
                SIZE_X = (image.shape[1]//patch_size)*patch_size #Nearest size divisible by our patch size
                SIZE_Y = (image.shape[0]//patch_size)*patch_size #Nearest size divisible by our patch size
                image = Image.fromarray(image)
                image = image.crop((0 ,0, SIZE_X, SIZE_Y))  #Crop from top left corner
                #image = image.resize((SIZE_X, SIZE_Y))  #Try not to resize for semantic segmentation
                image = np.array(image)             
       
                #Extract patches from each image
                print("Now patchifying image:", path+"/"+image_name)
                patches_img = patchify(image, (patch_size, patch_size, 3), step=patch_size)  #Step=256 for 256 patches means no overlap
        
                for i in range(patches_img.shape[0]):
                    for j in range(patches_img.shape[1]):
                        
                        single_patch_img = patches_img[i,j,:,:]
                        
                        #Use minmaxscaler instead of just dividing by 255. 
                        single_patch_img = scaler.fit_transform(single_patch_img.reshape(-1, single_patch_img.shape[-1])).reshape(single_patch_img.shape)
                        
                        #single_patch_img = (single_patch_img.astype('float32')) / 255. 
                        single_patch_img = single_patch_img[0] #Drop the extra unecessary dimension that patchify adds.                               
                        image_dataset.append(single_patch_img)
                

sat_Data/S2A_MSIL2A_20230627T073621_N0509_R092_T37NCC_20230627T110801.SAFE/GRANULE/L2A_T37NCC_A041845_20230627T075440/IMG_DATA/
sat_Data/S2A_MSIL2A_20230627T073621_N0509_R092_T37NCC_20230627T110801.SAFE/GRANULE/L2A_T37NCC_A041845_20230627T075440/IMG_DATA/R10m


/opt/conda/lib/python3.11/site-packages/PIL/Image.py:3176: DecompressionBombWarning: Image size (115605504 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Now patchifying image: sat_Data/S2A_MSIL2A_20230627T073621_N0509_R092_T37NCC_20230627T110801.SAFE/GRANULE/L2A_T37NCC_A041845_20230627T075440/IMG_DATA/R10m/T37NCC_20230627T073621_AOT_10m.jp2


/opt/conda/lib/python3.11/site-packages/PIL/Image.py:3176: DecompressionBombWarning: Image size (115605504 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Now patchifying image: sat_Data/S2A_MSIL2A_20230627T073621_N0509_R092_T37NCC_20230627T110801.SAFE/GRANULE/L2A_T37NCC_A041845_20230627T075440/IMG_DATA/R10m/T37NCC_20230627T073621_WVP_10m.jp2


...

Now patchifying image: sat_Data/S2A_MSIL2A_20230627T073621_N0509_R092_T37NCC_20230627T110801.SAFE/GRANULE/L2A_T37NCC_A041845_20230627T075440/IMG_DATA/R10m/T37NCC_20230627T073621_TCI_10m.jp2


/opt/conda/lib/python3.11/site-packages/PIL/Image.py:3176: DecompressionBombWarning: Image size (115605504 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Now patchifying image: sat_Data/S2A_MSIL2A_20230627T073621_N0509_R092_T37NCC_20230627T110801.SAFE/GRANULE/L2A_T37NCC_A041845_20230627T075440/IMG_DATA/R10m/T37NCC_20230627T073621_B08_10m.jp2


/opt/conda/lib/python3.11/site-packages/PIL/Image.py:3176: DecompressionBombWarning: Image size (115605504 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Now patchifying image: sat_Data/S2A_MSIL2A_20230627T073621_N0509_R092_T37NCC_20230627T110801.SAFE/GRANULE/L2A_T37NCC_A041845_20230627T075440/IMG_DATA/R10m/T37NCC_20230627T073621_B04_10m.jp2
sat_Data/S2A_MSIL2A_20230627T073621_N0509_R092_T37NCC_20230627T110801.SAFE/GRANULE/L2A_T37NCC_A041845_20230627T075440/IMG_DATA/R20m
sat_Data/S2A_MSIL2A_20230627T073621_N0509_R092_T37NCC_20230627T110801.SAFE/GRANULE/L2A_T37NCC_A041845_20230627T075440/IMG_DATA/R60m


In [ ]:
#image_dataset

In [15]:
 
 #Now do the same as above for masks
 #For this specific dataset we could have added masks to the above code as masks have extension png
mask_dataset = []  
for path, subdirs, files in os.walk(root_directory):
    #print(path)  
    dirname = path.split(os.path.sep)[-1]
    if dirname == 'masks':   #Find all 'images' directories
        masks = os.listdir(path)  #List of all image names in this subdirectory
        for i, mask_name in enumerate(masks):  
            if mask_name.endswith(".png"):   #Only read png images... (masks in this dataset)
               
                mask = cv2.imread(path+"/"+mask_name, 1)  #Read each image as Grey (or color but remember to map each color to an integer)
                mask = cv2.cvtColor(mask,cv2.COLOR_BGR2RGB)
                SIZE_X = (mask.shape[1]//patch_size)*patch_size #Nearest size divisible by our patch size
                SIZE_Y = (mask.shape[0]//patch_size)*patch_size #Nearest size divisible by our patch size
                mask = Image.fromarray(mask)
                mask = mask.crop((0 ,0, SIZE_X, SIZE_Y))  #Crop from top left corner
                #mask = mask.resize((SIZE_X, SIZE_Y))  #Try not to resize for semantic segmentation
                mask = np.array(mask)             
       
                #Extract patches from each image
                print("Now patchifying mask:", path+"/"+mask_name)
                patches_mask = patchify(mask, (patch_size, patch_size, 3), step=patch_size)  #Step=256 for 256 patches means no overlap
        
                for i in range(patches_mask.shape[0]):
                    for j in range(patches_mask.shape[1]):
                        
                        single_patch_mask = patches_mask[i,j,:,:]
                        #single_patch_img = (single_patch_img.astype('float32')) / 255. #No need to scale masks, but you can do it if you want
                        single_patch_mask = single_patch_mask[0] #Drop the extra unecessary dimension that patchify adds.                               
                        mask_dataset.append(single_patch_mask) 
 

In [ ]:
image_dataset = np.array(image_dataset)
mask_dataset =  np.array(mask_dataset)

In [6]:
image_dataset


array([], dtype=float64)